# Homework 6 — CSCI 347 Data Mining

**Name:** Lucas Jones

> Show your work. Include any code snippets that you used to generate answers. Complete this assignment individually. Unless otherwise stated, you can use code snippets to generate answers. If you are asked to do the calculations by hand, include your working.

In [1]:
import numpy as np
import pandas as pd
from fractions import Fraction

---
## Question 1 — Naïve Bayes (Categorical) [15 points]

Suppose you are asked to use Naïve Bayes to predict whether a person will go hiking or not, based on the following **training** dataset.

In [2]:
data = pd.DataFrame({
    'Weather':    ['Snow','Overcast','Sunny','Overcast','Overcast','Snow','Overcast','Sunny','Sunny','Snow',
                   'Snow','Overcast','Overcast','Sunny','Snow','Snow','Windy','Windy','Windy'],
    'Weekend':    ['Yes','No','Yes','Yes','No','No','Yes','Yes','No','No',
                   'Yes','Yes','No','No','Yes','No','No','Yes','Yes'],
    'Finished_HW':['No','No','No','Yes','Yes','Yes','No','No','Yes','Yes',
                   'No','No','Yes','Yes','Yes','No','No','No','Yes'],
    'Go_Hiking':  ['Yes','No','Yes','Yes','Yes','No','No','No','Yes','Yes',
                   'No','No','Yes','Yes','No','No','No','No','Yes']
})
data

,Weather,Weekend,Finished_HW,Go_Hiking
0,Snow,Yes,No,Yes
1,Overcast,No,No,No
2,Sunny,Yes,No,Yes
3,Overcast,Yes,Yes,Yes
4,Overcast,No,Yes,Yes
5,Snow,No,Yes,No
6,Overcast,Yes,No,No
7,Sunny,Yes,No,No
8,Sunny,No,Yes,Yes
9,Snow,No,Yes,Yes


### Part (a) — Prior Probabilities [4 points]

Find **P(c_i)**, the prior probabilities for the "Yes" and "No" classes, where c₁ = "Yes" and c₂ = "No".

__P(c1=Yes) = 9/19 = 0.47__

__P(c2=No) = 10/19 = 0.53__

### Part (b) — Likelihoods P(x|c₁) and P(x|c₂) [8 points]

Given the new data instance **x = (Sunny, Yes, No)**, find P(x|c₁) and P(x|c₂).

In [7]:
# Your work here
x = {'Weather': 'Sunny', 'Weekend': 'Yes', 'Finished_HW': 'No'}
print("New Instance:", x)


New Instance: {'Weather': 'Sunny', 'Weekend': 'Yes', 'Finished_HW': 'No'}


__P(x|c1) = (3/9)(4/9)(2/9) = 0.033__

__P(x|c2) = (1/10)(6/10)(8/10) = 0.048__

### Part (c) — Posterior Scores [2 points]

Compute **P(x|c₁)·P(c₁)** and **P(x|c₂)·P(c₂)**.

__P(x|c1)·P(c1) = 0.033 · 0.47 = 0.0156__

__P(x|c2)·P(c2) = 0.048 · 0.53= 0.0253__

### Part (d) — Classification [1 point]

Which class should the new data instance be assigned to?

__Since 0.0253 > 0.0156, x is classified as "No"/will not go hiking__

---
## Question 2 — Gaussian Naïve Bayes [4 points]

Use Gaussian Naïve Bayes for the following numerical training dataset. Predict the class of the new data instance **x**.

In [4]:
numerical_data = pd.DataFrame({
    'Rain_inches': [0, 0.5, 1, 5, 0.3, 0.4, 0.1, 0, 0, 3, 6, 2.1, 1.02],
    'Sleep_hours': [9, 5, 7, 7, 8, 4, 9, 9, 8, 10, 8, 8, 8.5],
    'HW_pct':      [80, 90, 95, 100, 100, 100, 27, 50, 100, 98, 95, 70, 98],
    'Go_Hiking':   ['Yes','No','Yes','Yes','Yes','No','No','No','Yes','Yes','No','No','Yes']
})
numerical_data

,Rain_inches,Sleep_hours,HW_pct,Go_Hiking
0,0.00,9.0,80,Yes
1,0.50,5.0,90,No
2,1.00,7.0,95,Yes
3,5.00,7.0,100,Yes
4,0.30,8.0,100,Yes
5,0.40,4.0,100,No
6,0.10,9.0,27,No
7,0.00,9.0,50,No
8,0.00,8.0,100,Yes
9,3.00,10.0,98,Yes


In [8]:
# New data instance
x_new = (0.58, 5.3, 75)
features = ['Rain_inches', 'Sleep_hours', 'HW_pct']

def gaussian_pdf(x, mean, std):
    return (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean) / std) ** 2)

yes = numerical_data[numerical_data['Go_Hiking'] == 'Yes']
no  = numerical_data[numerical_data['Go_Hiking'] == 'No']
n   = len(numerical_data)

score_yes = len(yes) / n
score_no  = len(no)  / n

for i, feat in enumerate(features):
    score_yes *= gaussian_pdf(x_new[i], yes[feat].mean(), yes[feat].std())
    score_no  *= gaussian_pdf(x_new[i], no[feat].mean(),  no[feat].std())

print(f"p(x|Yes) * P(Yes) = {score_yes:.10f}")
print(f"p(x|No)  * P(No)  = {score_no:.10f}")

p(x|Yes) * P(Yes) = 0.0000008210
p(x|No)  * P(No)  = 0.0001282006


__X is classified as No/will not go hiking. This is because P(x|Yes)·P(Yes) < P(x|No)·P(No)__

---
## Question 3 — Association Rule Mining [11 points]

Consider the following data that shows transactions of items purchased in a supermarket.

In [6]:
transactions = {
    1:  ['Toilet paper', 'Beans', 'Rice', 'Milk', 'Baby wipes', 'Diapers'],
    2:  ['Oat milk', 'Beans', 'Toilet paper', 'Orange juice', 'Bread'],
    3:  ['Oat milk', 'Milk', 'Orange juice', 'Toilet paper', 'Bread'],
    4:  ['Beans', 'Toilet paper', 'Baby wipes', 'Diapers'],
    5:  ['Toilet paper', 'Butter', 'Baby wipes', 'Diapers'],
    6:  ['Milk', 'Toilet paper', 'Bread'],
    7:  ['Milk', 'Rice', 'Bread'],
    8:  ['Beans', 'Milk', 'Rice', 'Toilet paper'],
    9:  ['Milk', 'Butter', 'Diapers', 'Bread'],
    10: ['Beans', 'Rice', 'Toilet paper'],
}

for tid, items in transactions.items():
    print(f"T{tid:2d}: {items}")

T 1: ['Toilet paper', 'Beans', 'Rice', 'Milk', 'Baby wipes', 'Diapers']
T 2: ['Oat milk', 'Beans', 'Toilet paper', 'Orange juice', 'Bread']
T 3: ['Oat milk', 'Milk', 'Orange juice', 'Toilet paper', 'Bread']
T 4: ['Beans', 'Toilet paper', 'Baby wipes', 'Diapers']
T 5: ['Toilet paper', 'Butter', 'Baby wipes', 'Diapers']
T 6: ['Milk', 'Toilet paper', 'Bread']
T 7: ['Milk', 'Rice', 'Bread']
T 8: ['Beans', 'Milk', 'Rice', 'Toilet paper']
T 9: ['Milk', 'Butter', 'Diapers', 'Bread']
T10: ['Beans', 'Rice', 'Toilet paper']


### Part (a) — Support of {Milk, Toilet paper} [2 points]

What is the support of the itemset **{Milk, Toilet paper}**?

__4/10 = 0.4__

### Part (b) — Frequent Itemsets of Size 2 (minsup = 4) [3 points]

If we use the minimum support threshold of **4**, what are the itemsets of size 2 that are frequent?

__{Beans, Toilet Paper}(5 occurences)__

__{Milk, Bread}(4 occurences)__

__{Milk, Toilet Paper}(4 occurences)__

### Part (c) — A-Priori Candidate Itemsets of Size 3 (minsup = 4) [4 points]

If we use the **A-Priori algorithm** to generate frequent itemsets using a minsup of 4, what is the set of **candidate itemsets of size 3** that will be generated by the algorithm?

__{Beans, Toilet Paper} + {Milk, Toilet Paper} -> {Beans, Milk, Toilet Paper} is pruned because {Beans, Milk} is not frequent.__

__{Milk, Bread} + {Milk, Toilet Paper} -> {Milk, Bread, Toilet Papper} is pruned because {Bread, Toilet Paper} is not frequent__

__Therefore C_3 = empty set__

### Part (d) — Confidence & Lift of rice → beans [2 points]

What is the **confidence** and **lift** of the rule **rice → beans**?

__Confidence = 3/4 = 0.75__

__Lift = 0.75/0.5 = 1.5__

### Part (e) — Confidence & Lift of beans → rice [1 point]

What is the **confidence** and **lift** of the rule **beans → rice**?

__Confidence = 3/5 = 0.6__

__Lift = 0.6/0.4 = 1.5__